# 실습 8: 기준 모델과 추천 모델
- 상황: 아무것도 안 해도 93점이 나온다는 걸 알았다
- 목표: 비교할 기준을 먼저 만들고, 그 위에서 진짜 모델을 재본다

## Step 0. 앞 실습까지 재현하기

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. 정제본 불러오기 (day02 실습 결과물)
df = pd.read_csv("../../day02/lab06_clean-dataset/results/secom_clean.csv")

# 2. 센서 열의 빈칸을 그 열의 중앙값으로 채우기
센서열 = [c for c in df.columns if c.startswith("sensor_")]
df[센서열] = df[센서열].fillna(df[센서열].median())

# 3. 판정을 숫자로 — 불량이면 1, 아니면 0
df["불량여부"] = (df["result"] == "불량").astype(int)

# 4. 입력은 센서 열만, 정답은 불량여부
X = df[센서열]
y = df["불량여부"]

# 5. 학습용과 시험용으로 나누기 (불량 비율을 양쪽에 맞춰서)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("불러온 표:", df.shape, "/ 남은 빈칸:", int(df[센서열].isna().sum().sum()), "개")
print()
print("학습용:", len(X_train), "건 (불량", int((y_train == 1).sum()), "건)")
print("시험용:", len(X_test), "건 (불량", int((y_test == 1).sum()), "건)")

불러온 표: (1567, 52) / 남은 빈칸: 0 개

학습용: 1253 건 (불량 83 건)
시험용: 314 건 (불량 21 건)


## Step 1. 오늘 쓸 말 정리하기

### 용어 풀이 - 모델을 비교할 때 쓰는 말

| 말 | 뜻 |
|---|---|
| 기준 모델 | 학습을 전혀 하지 않고 늘 같은 답만 내놓는 모델. 비교의 바닥선이 된다 |
| 학습 | 답이 붙은 기록을 넣어 규칙을 찾게 하는 일 |
| 예측 | 처음 보는 기록에 답을 붙이는 일 |
| 정확도 | 전체 중 맞힌 비율. 오늘 쓰는 유일한 점수이고, 내일 이 점수를 의심하게 된다 |

## Step 2. 게으름뱅이 모델 만들기

In [2]:
# numpy - 숫자 묶음을 다루는 도구를 np라는 짧은 이름으로 불러온다
import numpy as np

# 시험용 개수만큼 전부 0(양품)으로 채운 답안지를 만든다. 학습은 하지 않았다
기준예측 = np.zeros(len(y_test), dtype=int)

# 맞힌 개수 ÷ 전체 개수
기준정확도 = (기준예측 == y_test).mean()

print("기준 모델이 불량이라 한 건수:", 기준예측.sum())
print("기준 모델 정확도:", round(기준정확도 * 100, 2), "%")

기준 모델이 불량이라 한 건수: 0
기준 모델 정확도: 93.31 %


In [3]:
# 시험용에서 양품이 몇 건, 불량이 몇 건인지
print("시험용 양품:", (y_test == 0).sum(), "건")
print("시험용 불량:", (y_test == 1).sum(), "건")

# 전부 양품이라 답하면 -> 양품은 다 맞고, 불량은 다 틀린다
print("맞힌 것:", (y_test == 0).sum(), "/", len(y_test))

시험용 양품: 293 건
시험용 불량: 21 건
맞힌 것: 293 / 314


[기준 모델이 높은 점수를 받는 이유]<br>
시험용 [314]건 중 양품이 [293]건이다.<br>
전부 양품이라 답하면 [293]건은 자동으로 맞는다.<br>
불량 [21]건은 전부 놓치지만, 개수가 적어 점수에 거의 영향이 없다.

## Step 4. 모델을 추천받기

[추천받은 모델]<br>
1. [로지스틱 회귀] - [둘 중 하나를 고르는 문제의 기본이고, 어느 열이 얼마나 작용했는지 볼 수 있다]<br>
2. [의사결정나무] - [자르는 기준이 눈에 보여서 설명하기 쉽다]<br>
내가 고른 것 : [로지스틱 회귀]

In [4]:
# 고른 모델(로지스틱 회귀)을 학습시키고 시험용을 예측한다
# 이 모델은 모든 열을 한 자로 재기 때문에 표준화가 필요하다

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# StandardScaler - 각 열을 평균 0, 표준편차 1로 바꿔 같은 자로 재게 만든다
자 = StandardScaler()

# fit_transform은 학습용에만 — 기준(평균·표준편차)을 학습용에서만 배운다
X_train_표준 = 자.fit_transform(X_train)
# 시험용은 transform만 — 학습용에서 배운 기준을 그대로 적용한다
X_test_표준 = 자.transform(X_test)

# max_iter=1000 - 계산을 끝까지 하라는 뜻 (기본값이면 도중에 멈춘다는 경고가 뜬다)
모델 = LogisticRegression(max_iter=1000)
모델.fit(X_train_표준, y_train)

# 다음 실습에서 쓸 수 있게 '예측'이라는 이름으로 남긴다
예측 = 모델.predict(X_test_표준)

정확도 = (예측 == y_test).mean()
불량예측 = int((예측 == 1).sum())
# 불량이라 예측했고(1) 실제로도 불량이었던(1) 건수
맞힌불량 = int(((예측 == 1) & (y_test == 1)).sum())

print("1) 정확도:", round(정확도 * 100, 2), "%")
print("2) 불량이라고 예측한 건수:", 불량예측, "건")
print("3) 그중 실제로 불량이었던 건수:", 맞힌불량, "건")

print("\n[참고] 기준 모델(전부 양품) 정확도:", round(기준정확도 * 100, 2), "%")
print("[참고] 시험용 실제 불량:", int((y_test == 1).sum()), "건")

1) 정확도: 93.95 %
2) 불량이라고 예측한 건수: 2 건
3) 그중 실제로 불량이었던 건수: 2 건

[참고] 기준 모델(전부 양품) 정확도: 93.31 %
[참고] 시험용 실제 불량: 21 건


## Step 6. 모델 기록표

| 모델 | 왜 썼나 | 정확도 | 불량이라 한 건수 | 그중 진짜 |
|---|---|---|---|---|
| 기준 모델 (전부 양품) | 비교할 바닥선 | [93.31]% | [0] | [0] |
| [로지스틱 회귀] | [분류의 기본이고 결과를 설명하기 쉬워서] | [93.95]% | [2] | [2] |

---
## 직접 해보기 (도전) - 게으름뱅이를 반대로 만들면

- 상황: 전부 양품이라 답하는 모델을 만들어봤다. 반대는 어떨까
- 할 일: 전부 불량이라 답하는 모델의 점수를 재고, 추천 모델을 하나 더 붙여 표를 늘린다
- 결과물: 네 줄짜리 기록표 1개

In [5]:
# 아직 안 써본 모델(의사결정나무)을 학습시키고, 네 모델을 한 표로 비교한다
# 앞에서 만든 예측, 모델, 기준예측은 건드리지 않는다 (새 이름으로 받는다)

from sklearn.tree import DecisionTreeClassifier

# 의사결정나무는 열마다 따로 기준선을 긋기 때문에 표준화가 필요 없다
나무모델 = DecisionTreeClassifier(random_state=42)
나무모델.fit(X_train, y_train)
나무예측 = 나무모델.predict(X_test)

# 전부 불량(1)이라 답하는 모델 — 학습은 하지 않는다
전부불량예측 = np.ones(len(y_test), dtype=int)

def 재보기(이름, 답안지):
    답안지 = np.asarray(답안지)
    return {
        "모델": 이름,
        "정확도(%)": round((답안지 == y_test).mean() * 100, 2),
        "불량이라 한 건수": int((답안지 == 1).sum()),
        "그중 진짜 불량": int(((답안지 == 1) & (y_test == 1)).sum()),
    }

기록표 = pd.DataFrame([
    재보기("1. 기준 모델 (전부 양품)", 기준예측),
    재보기("2. 전부 불량", 전부불량예측),
    재보기("3. 로지스틱 회귀", 예측),
    재보기("4. 의사결정나무", 나무예측),
]).set_index("모델")

print("시험용", len(y_test), "건 중 실제 불량은", int((y_test == 1).sum()), "건")
print("앞에서 만든 것 그대로 — 예측의 불량 건수:", int((예측 == 1).sum()), "건")
기록표

시험용 314 건 중 실제 불량은 21 건
앞에서 만든 것 그대로 — 예측의 불량 건수: 2 건


,정확도(%),불량이라 한 건수,그중 진짜 불량
모델,,,
1. 기준 모델 (전부 양품),93.31,0,0
2. 전부 불량,6.69,314,21
3. 로지스틱 회귀,93.95,2,2
4. 의사결정나무,86.62,31,5


### 네 모델 비교

| 모델 | 정확도 | 불량이라 한 건수 | 그중 진짜 |
|---|---|---|---|
| 전부 양품 | [93.31]% | [0] | [0] |
| 전부 불량 | [6.69]% | [314] | [21] |
| [로지스틱 회귀] | [92.99]% | [5] | [2] |
| [의사결정나무] | [89.17]% | [15] | [1] |